In [1]:
import requests

url = "https://www.weather-atlas.com/en/new-york-usa/new-york-climate"

headers = {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64; rv:148.0) Gecko/20100101 Firefox/148.0",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.weather-atlas.com/",
}

cookies = {
    "weather_units": "c|mm|mb|km",
}

response = requests.get(
    url,
    headers=headers,
    cookies=cookies,
    timeout=30,
)

print("status:", response.status_code)
print("final url:", response.url)
print(response.text[:3000])


status: 200
final url: https://www.weather-atlas.com/en/new-york-usa/new-york-climate
<!doctype html><html lang="en-us"><head><script>
// templated by golang 
function __setCMPv2RequestData() {
    window._CMPv2RequestData = {
	    "language": "en",
	    "stylingLogo": "//g.ezodn.com/utilcave_com/middleton/img.webp?dirname=weather_atlas_com&amp;img=/logo/weather_atlas_com"
    };
}
__setCMPv2RequestData();

var gtagLoadBackoff = 50;
function gtagLoadedCheck() {
    if(typeof gtag == 'undefined') {
        gtagLoadBackoff += 50;

        return setTimeout(function(){
            gtagLoadedCheck();
        }, gtagLoadBackoff);
    } else {
        gtag('consent', 'default', {
            'ad_storage': 'denied',
            'ad_user_data': 'denied',
            'ad_personalization': 'denied',
            'analytics_storage': 'denied'
        });
    }
}

gtagLoadedCheck();</script>

<script src="https://privacy.gatekeeperconsent.com/tcf2_stub.js" data-cfasync="false"></script><script>var 

In [2]:
import re
import requests
import pandas as pd
from bs4 import BeautifulSoup


def construct_url(country, state, city, use_query_units=False):
    if country == "United States":
        country = "".join([state.replace(" ", "-"), "-usa"])
    else:
        country = country.replace(" ", "-")

    city = city.replace(" ", "-")
    base_url = f"https://www.weather-atlas.com/en/{country}/{city}-climate"

    # Я бы по умолчанию уже не полагался на query string.
    # Но для сравнения можно включать use_query_units=True.
    if use_query_units:
        return f"{base_url}?c,mm,mb,km"
    return base_url


def scrap_city_dict(url, session=None):
    city_dict = {}
    session = session or requests.Session()

    response = session.get(url, allow_redirects=True, timeout=30)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "lxml")
    ul_tags = soup.find_all("ul", class_="list-unstyled mb-0")

    if not ul_tags:
        print(f"wrong url or blocked page: {url}")
        print("final url:", response.url)
        print(response.text[:1000])
        return city_dict

    for ul in ul_tags:
        li_tags = ul.find_all("li")
        for li in li_tags:
            if li.a and li.span:
                city_dict[li.a.text] = li.span.text

    for key, value in city_dict.items():
        city_dict[key] = re.split(r"(\-?\d*\.?\d+|\d+)", value)

    return city_dict


def get_params_dict(city_dict):
    params_dict = {}
    for key, value in city_dict.items():
        item = key.split(" in ")
        param = item[0]
        if param not in params_dict:
            params_dict[param] = [
                param.removeprefix("Average ")
                .removesuffix("erature")
                .replace(" ", "_")
                .lower()
            ]
            params_dict[param].append(
                value[2]
                .strip()
                .lstrip("В")
                .replace("h", "hours")
                .replace("km/hours", "km/h")
            )
    return params_dict


def get_months_dict(city_dict):
    count = 0
    months_dict = {}
    for key, value in city_dict.items():
        item = key.split(" in ")
        month = item[1]
        if month not in months_dict:
            count += 1
            months_dict[month] = count
    return months_dict


def get_columns_list(params_dict):
    return [value[0] for value in params_dict.values()]


def params_template_df(months_dict, columns_list):
    df_params_templ = pd.DataFrame(
        index=months_dict.values(), columns=columns_list, dtype=None
    )
    df_params_templ.loc[:, :] = None
    df_params_templ.insert(0, "city_id", None)
    df_params_templ.insert(1, "month", df_params_templ.index)
    return df_params_templ


def fill_params_template_df(city_dict, months_dict, params_dict, df_params_fill):
    for key, value in city_dict.items():
        item = key.split(" in ")
        month = months_dict[item[1]]
        column = params_dict[item[0]][0]
        num = float(value[1])
        df_params_fill.loc[month, column] = num
    return df_params_fill


# --- тестовая сессия с cookie Celsius ---
session = requests.Session()
session.headers.update(
    {
        "User-Agent": "Mozilla/5.0 (X11; Linux x86_64; rv:148.0) Gecko/20100101 Firefox/148.0",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.9",
        "Referer": "https://www.weather-atlas.com/",
    }
)
session.cookies.set("weather_units", "c|mm|mb|km", domain="www.weather-atlas.com")

country = "United States"
state = "New York"
city = "New York"

# 1) рекомендованный URL: без query string
url = construct_url(country, state, city, use_query_units=False)

# 2) если хочешь сравнить, можно отдельно протестировать и с query string:
# url = construct_url(country, state, city, use_query_units=True)

print("URL:", url)
print("cookies:", session.cookies.get_dict())

city_dict = scrap_city_dict(url, session=session)

print("parsed items:", len(city_dict))
print("first 10 keys:")
for i, key in enumerate(city_dict.keys()):
    if i == 10:
        break
    print("-", key, "=>", city_dict[key])

if city_dict:
    params_dict = get_params_dict(city_dict)
    months_dict = get_months_dict(city_dict)
    columns_list = get_columns_list(params_dict)
    df_params_empty = params_template_df(months_dict, columns_list)
    df_params_empty["city_id"] = (
        5128581  # можно поставить любой тестовый id для New York
    )
    df_result = fill_params_template_df(
        city_dict, months_dict, params_dict, df_params_empty
    )

    print("\nparams_dict:")
    for k, v in list(params_dict.items())[:10]:
        print(k, "=>", v)

    print("\nmonths_dict:")
    print(months_dict)

    print("\nresult dataframe:")
    display(df_result)
else:
    print("city_dict is empty")


URL: https://www.weather-atlas.com/en/New-York-usa/New-York-climate
cookies: {'weather_units': 'c|mm|mb|km'}
parsed items: 192
first 10 keys:
- Average high temperature in January => ['', '2.5', '°C']
- Average high temperature in February => ['', '4', '°C']
- Average high temperature in March => ['', '8', '°C']
- Average high temperature in April => ['', '14.2', '°C']
- Average high temperature in May => ['', '20', '°C']
- Average high temperature in June => ['', '24.8', '°C']
- Average high temperature in July => ['', '28.3', '°C']
- Average high temperature in August => ['', '27.7', '°C']
- Average high temperature in September => ['', '23.8', '°C']
- Average high temperature in October => ['', '17.4', '°C']

params_dict:
Average high temperature => ['high_temp', '°C']
Average low temperature => ['low_temp', '°C']
Average pressure => ['pressure', 'mbar']
Average wind speed => ['wind_speed', 'km/h']
Average humidity => ['humidity', '%']
Average rainfall => ['rainfall', 'mm']
Average 

,city_id,month,high_temp,low_temp,pressure,wind_speed,humidity,rainfall,rainfall_days,snowfall,snowfall_days,sea_temp,daylight,sunshine,sunshine_days,uv_index,cloud_cover,visibility
1,5128581,1,2.5,-3.3,1018.0,16.0,74.0,55.0,9.1,40.0,5.8,5.4,9.0,5.0,17.9,2.0,43.0,9.0
2,5128581,2,4.0,-2.4,1017.1,15.6,76.0,60.0,10.8,48.0,5.5,4.3,10.0,4.0,13.7,2.0,47.0,9.0
3,5128581,3,8.0,1.2,1017.7,15.7,73.0,72.0,12.7,67.0,4.1,4.5,12.0,6.0,14.3,2.0,47.0,9.0
4,5128581,4,14.2,6.7,1015.3,14.8,72.0,76.0,15.7,5.0,0.4,7.3,13.0,8.0,12.6,4.0,45.0,9.0
5,5128581,5,20.0,12.3,1016.3,12.3,75.0,86.0,17.9,0.0,0.0,11.4,14.0,8.0,11.0,5.0,44.0,9.0
6,5128581,6,24.8,17.1,1013.6,11.5,74.0,85.0,17.4,0.0,0.0,18.1,15.0,9.0,9.8,6.0,36.0,9.0
7,5128581,7,28.3,20.8,1014.4,10.7,72.0,102.0,18.5,0.0,0.0,22.3,14.0,10.0,10.2,6.0,31.0,9.0
8,5128581,8,27.7,20.4,1015.4,10.8,70.0,94.0,16.9,0.0,0.0,23.4,13.0,10.0,11.4,6.0,30.0,9.0
9,5128581,9,23.8,16.8,1017.7,11.8,73.0,62.0,14.3,0.0,0.0,21.2,12.0,8.0,13.1,5.0,34.0,9.0
10,5128581,10,17.4,11.0,1016.6,13.9,75.0,80.0,13.7,0.0,0.0,17.3,11.0,5.0,15.5,3.0,41.0,9.0
